【最終課題】写真に写る動物が犬か猫かを分類しよう

In [ ]:
"""
最終課題用のノートブックのなかに、犬と猫の画像を学習したモデルを作成して、
分類を行なうプログラムを作成してください。

本レッスン内容で学習した流れに沿って、深層学習プログラムを作成してください
データの前処理や水増しの処理を入れてください
MobileNetV2 のモデルを利用してください（画像サイズは MobileNetV2 が
対応する大きさへのリサイズが必要です）
必ず最後に evaluate() を実行して、正答率がわかるようにしてください。
"""

In [9]:
#犬と猫の画像を読み込み　画像サイズ75×75　train70枚、test50枚

import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(75, 75), #MobileNetV2のモデルで利用できるサイズ
    label_mode="binary", #2値分類（binary）：あるかないか、の2値だから
    batch_size=32, #”1回の学習で”何枚の画像を使うか。デフォルトは32
    shuffle=True #読み込み順。学習が偏らないようシャッフルして利用
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(75, 75),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

# いったんデータを表示してみる ※image_dataset_from_directory() はそのまま見れない
#　list(train_dataset.as_numpy_iterator())[0]

# 分類名＊フォルダ名（dog／cat）をリストとして格納
class_names = train_dataset.class_names
class_names

# 画像を水増しする。　そのための関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

# 水増し処理の実行
# map()は非破壊メソッド。新しい変数へ要格納
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

# 水増ししたデータを訓練データに追加（連結）し、最後にシャッフル
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)
train_dataset = train_dataset.shuffle(32)

"""
#訓練データの一部を表示していったん確認（課題外）
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))

for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i].numpy().astype("uint8")[0]])
"""

#MobileNetV2 の学習済みモデルで転移学習
# MobileNetV2モデル作成
input_layer = tf.keras.Input(shape=(75, 75, 3))   # 入力層
# 入力層と正規化する層を結合
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer) 
  # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(75, 75, 3),#ピクセル数タテ、ヨコ、色RGB(3)
    input_tensor=l_layer, #入力層（1レイヤーめ）を指定
    include_top=False, #出力層を自分で作成するためFalse。転移学習を可能に
    weights="imagenet", #mageNet(大規模データセット)で学習した重みを利用
    pooling='avg' #プーリング層(畳み込み層)の出力を縮小するための層の計算手法
)
base_model.trainable = False #出力層でのみ、入力された画像を学習

# Dense層（出力層）を追加
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')
#出力層のノード数は0か1の一種類なので１

# base_modelに出力層（Dense層）を追加したモデル作成
model = tf.keras.Sequential([
    base_model,
    output_layer
])

# さいごにcompileする
model.compile(optimizer="adam", #今回のケースは adam という最適化手法
              loss='binary_crossentropy', #損失関数。正解との差を計算
              metrics=["accuracy"]) #評価関数。正解率のみ指定

# できたmodelに学習させる　（20回）
model.fit(train_dataset, epochs=20) #epochはデータセットの学習回数

# テストデータで分類を実行
pred_data = model.predict(test_dataset)

# 分類した結果を確認する。（課題外）
#pred_data

# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)



Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.


C:\Users\user\AppData\Local\Temp\ipykernel_18648\1874087128.py:84: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(


Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 23s 186ms/step - accuracy: 0.7388 - loss: 0.5607
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 172ms/step - accuracy: 0.8892 - loss: 0.2326
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 175ms/step - accuracy: 0.9369 - loss: 0.1614
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 179ms/step - accuracy: 0.9527 - loss: 0.1298
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 178ms/step - accuracy: 0.9546 - loss: 0.1174
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 187ms/step - accuracy: 0.9459 - loss: 0.1348
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 11s 170ms/step - accuracy: 0.9652 - loss: 0.1046
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 11s 170ms/step - accuracy: 0.9772 - loss: 0.0807
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 11s 169ms/step - accuracy: 0.9810 - loss: 0.0723
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 180ms/step - accuracy: 0.9819 - loss: 0.0645
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 13s 201ms/step - accuracy: 0.9869 - loss: 0.0539
Epoch 12/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12

[0.2832423746585846, 0.9300000071525574]